<table width="100%">
<tr>
<td width="50%" align="left"><a href="./03_RQ3_expenditure_resources.ipynb">← Previous: RQ3 — Expenditure Resources</a></td>
<td width="50%" align="right"><a href="./05_RQ5_healthcare_outcomes.ipynb">Next: RQ5 — Healthcare Outcomes →</a></td>
</tr>
</table>

## **4.0 Healthcare Access**

#### This section analyses how access (wait-times) to healthcare vary across Canadian provinces as well as the relationship between healthcare resources and capacity. 

> - #### **Research Question: How does access to healthcare vary across Canadian provinces and territories, and how is access related to healthcare resources and capacity?**

In [ ]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
from dotenv import load_dotenv

import os

import matplotlib.pyplot as plt

In [ ]:
import pandas as pd
from daytascape_db_core.connection import get_db_engine

In [ ]:
DATABASE = "health_system_performance"

engine = get_db_engine(DATABASE)

print("Connected database:", engine.url.database)

Connected database: health_system_performance


In [ ]:
query = """
SELECT *
FROM master.analytics_province_year;
"""

df = pd.read_sql(query, engine)

print("Connected successfully to database!")

df.head(10)

Connected successfully to database!


,province_id,province_code,province_name,data_year,population,population_65_plus,population_80_plus,population_85_plus,population_65_share_pct,population_80_share_pct,...,private_health_expenditure_constant_2010_per_capita,provincial_government_health_expenditure_current,provincial_government_health_expenditure_per_capita,provincial_government_health_expenditure_constant_2010,provincial_government_health_expenditure_constant_2010_per_capi,territorial_government_health_expenditure_current,territorial_government_health_expenditure_per_capita,territorial_government_health_expenditure_constant_2010,territorial_government_health_expenditure_constant_2010_per_cap,nurses_total
0,15,PE,Prince Edward Island,2024,179709.0,36849.0,8218.0,3886.0,20.504816,4.572948,...,1914.110532,1.228779e+09,6881.989110,8.254174e+08,4622.892113,NaN,NaN,NaN,NaN,NaN
1,6,AB,Alberta,1975,1808689.0,134156.0,27442.0,12282.0,7.417306,1.517232,...,663.549287,6.949125e+08,384.207843,3.428803e+09,1895.739423,NaN,NaN,NaN,NaN,NaN
2,7,BC,British Columbia,1975,2499564.0,236960.0,51071.0,23239.0,9.480053,2.043196,...,679.794364,9.281825e+08,371.337761,4.379023e+09,1751.914626,NaN,NaN,NaN,NaN,NaN
3,7,BC,British Columbia,2022,5358845.0,1054174.0,250939.0,124925.0,19.671664,4.682707,...,2273.051979,3.010337e+10,5618.936230,2.317900e+10,4326.469070,NaN,NaN,NaN,NaN,108708.0
4,9,NB,New Brunswick,2021,790802.0,178652.0,39837.0,19193.0,22.591243,5.037544,...,2191.698134,3.945472e+09,4989.203072,3.151362e+09,3985.020465,NaN,NaN,NaN,NaN,NaN
5,5,CA,Canada,1975,23143275.0,1957075.0,377982.0,160362.0,8.456344,1.633226,...,568.363206,8.709289e+09,376.320519,4.111742e+10,1776.646681,NaN,NaN,NaN,NaN,NaN
6,8,MB,Manitoba,1975,1024975.0,104215.0,22287.0,9920.0,10.167565,2.174394,...,494.535967,3.766992e+08,367.520427,1.852127e+09,1806.997401,NaN,NaN,NaN,NaN,NaN
7,14,ON,Ontario,1980,8746013.0,850356.0,165961.0,70416.0,9.722785,1.897562,...,776.538159,5.164564e+09,590.505011,1.629120e+10,1862.700006,NaN,NaN,NaN,NaN,NaN
8,16,QC,Quebec,2020,8551095.0,1673371.0,401828.0,204785.0,19.569084,4.699141,...,1755.558556,4.704125e+10,5501.196296,3.642464e+10,4259.646558,NaN,NaN,NaN,NaN,NaN
9,14,ON,Ontario,2016,13876500.0,2260652.0,593482.0,302656.0,16.291226,4.276885,...,2000.807682,5.655294e+10,4075.446981,5.104966e+10,3678.856840,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['province_id', 'province_code', 'province_name', 'data_year',
       'population', 'population_65_plus', 'population_80_plus',
       'population_85_plus', 'population_65_share_pct',
       'population_80_share_pct', 'population_85_share_pct',
       'hospital_beds_total', 'current_hospital_beds_total',
       'physicians_total', 'registered_nurses', 'licensed_practical_nurses',
       'nurse_practitioners', 'registered_psychiatric_nurses', 'icu_beds',
       'long_term_care_beds', 'mental_health_addictions_beds',
       'obstetrics_beds', 'other_acute_care_beds', 'pediatrics_beds',
       'rated_capacity_beds', 'rehabilitation_beds',
       'bladder_cancer_surgery_median',
       'bladder_cancer_surgery_90th_percentile',
       'breast_cancer_surgery_median', 'breast_cancer_surgery_90th_percentile',
       'cabg_median', 'cabg_90th_percentile', 'cataract_surgery_median',
       'cataract_surgery_90th_percentile', 'colorectal_cancer_surgery_

In [ ]:
access_columns = [
    "province_code",
    "province_name",
    "data_year",
    "bladder_cancer_surgery_median",
    "breast_cancer_surgery_median",
    "cabg_median",
    "cataract_surgery_median",
    "colorectal_cancer_surgery_median",
    "ct_scan_median",
    "hip_fracture_repair_median",
    "hip_fracture_repair_emergency_inpatient_median",
    "hip_replacement_median",
    "knee_replacement_median",
    "lung_cancer_surgery_median",
    "mri_scan_median",
    "prostate_cancer_surgery_median",
    "radiation_therapy_median",
    "bladder_cancer_surgery_90th_percentile",
    "breast_cancer_surgery_90th_percentile",
    "cabg_90th_percentile",
    "cataract_surgery_90th_percentile",
    "colorectal_cancer_surgery_90th_percentile",
    "ct_scan_90th_percentile",
    "hip_fracture_repair_90th_percentile",
    "hip_fracture_repair_emergency_inpatient_90th_percentile",
    "hip_replacement_90th_percentile",
    "knee_replacement_90th_percentile",
    "lung_cancer_surgery_90th_percentile",
    "mri_scan_90th_percentile",
    "prostate_cancer_surgery_90th_percentile",
    "radiation_therapy_90th_percentile",
    "joint_replacement_cases",
    "joint_replacement_pct_within_benchmark",
    "hip_fracture_surgery_risk_adjusted_rate"
]

df_access = df[access_columns].copy()

In [ ]:
df_access

,province_code,province_name,data_year,bladder_cancer_surgery_median,breast_cancer_surgery_median,cabg_median,cataract_surgery_median,colorectal_cancer_surgery_median,ct_scan_median,hip_fracture_repair_median,...,hip_fracture_repair_emergency_inpatient_90th_percentile,hip_replacement_90th_percentile,knee_replacement_90th_percentile,lung_cancer_surgery_90th_percentile,mri_scan_90th_percentile,prostate_cancer_surgery_90th_percentile,radiation_therapy_90th_percentile,joint_replacement_cases,joint_replacement_pct_within_benchmark,hip_fracture_surgery_risk_adjusted_rate
0,PE,Prince Edward Island,2024,20.2,28.0,NaN,352.6,19.5,18.1,18.283333,...,48.8,631.5,731.5,NaN,652.0,NaN,27.0,324.0,44.0,79.2
1,AB,Alberta,1975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,BC,British Columbia,1975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BC,British Columbia,2022,25.0,20.0,10.0,44.0,21.0,24.0,28.550000,...,NaN,695.0,785.0,69.0,152.0,123.0,35.0,486.0,65.0,72.3
4,NB,New Brunswick,2021,27.0,23.0,10.0,71.0,23.0,NaN,21.683333,...,NaN,511.0,552.0,76.0,NaN,76.0,28.0,983.0,63.0,83.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
725,SK,Saskatchewan,2015,16.0,18.0,4.0,35.0,16.0,21.0,25.000000,...,NaN,141.0,142.0,27.0,152.0,138.0,20.0,NaN,NaN,NaN
726,QC,Quebec,2019,26.0,20.0,NaN,42.0,21.0,NaN,NaN,...,NaN,377.0,399.0,58.0,NaN,95.0,NaN,731.0,99.0,NaN
727,NU,Nunavut,1995,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
728,NU,Nunavut,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
id_cols = ["province_code", "province_name","data_year"]

wait_time_columns = [
    col for col in df_access.columns
    if col.endswith("_median") or col.endswith("_90th_percentile")
]

df_access[wait_time_columns].notna().sum().sort_values()

hip_fracture_repair_emergency_inpatient_median              61
hip_fracture_repair_emergency_inpatient_90th_percentile     61
mri_scan_median                                            116
ct_scan_median                                             116
ct_scan_90th_percentile                                    116
mri_scan_90th_percentile                                   116
prostate_cancer_surgery_90th_percentile                    120
prostate_cancer_surgery_median                             120
lung_cancer_surgery_median                                 125
lung_cancer_surgery_90th_percentile                        125
colorectal_cancer_surgery_90th_percentile                  131
bladder_cancer_surgery_90th_percentile                     131
bladder_cancer_surgery_median                              131
colorectal_cancer_surgery_median                           131
breast_cancer_surgery_90th_percentile                      142
breast_cancer_surgery_median                           

In [ ]:
latest_wait_year = {
    col: df_access.loc[df_access[col].notna(), "data_year"].max()
    for col in wait_time_columns
}

latest_wait_year

{'bladder_cancer_surgery_median': np.int64(2025),
 'breast_cancer_surgery_median': np.int64(2025),
 'cabg_median': np.int64(2025),
 'cataract_surgery_median': np.int64(2025),
 'colorectal_cancer_surgery_median': np.int64(2025),
 'ct_scan_median': np.int64(2025),
 'hip_fracture_repair_median': np.int64(2025),
 'hip_fracture_repair_emergency_inpatient_median': np.int64(2025),
 'hip_replacement_median': np.int64(2025),
 'knee_replacement_median': np.int64(2025),
 'lung_cancer_surgery_median': np.int64(2025),
 'mri_scan_median': np.int64(2025),
 'prostate_cancer_surgery_median': np.int64(2025),
 'radiation_therapy_median': np.int64(2025),
 'bladder_cancer_surgery_90th_percentile': np.int64(2025),
 'breast_cancer_surgery_90th_percentile': np.int64(2025),
 'cabg_90th_percentile': np.int64(2025),
 'cataract_surgery_90th_percentile': np.int64(2025),
 'colorectal_cancer_surgery_90th_percentile': np.int64(2025),
 'ct_scan_90th_percentile': np.int64(2025),
 'hip_fracture_repair_90th_percentile': 

### **4.1 Healthcare Wait-times Variation Accross Provinces**

In [ ]:
df_access_2025 = df_access[
    (df_access["data_year"] == 2025) &
    (df_access["province_code"].isin(
        ["AB", "BC", "MB", "NB", "NL", "NS", "ON", "PE", "QC", "SK"]
    ))
][
    ["province_code", "province_name", "data_year"] + wait_time_columns
].copy()

In [ ]:
df_access_2025

,province_code,province_name,data_year,bladder_cancer_surgery_median,breast_cancer_surgery_median,cabg_median,cataract_surgery_median,colorectal_cancer_surgery_median,ct_scan_median,hip_fracture_repair_median,...,colorectal_cancer_surgery_90th_percentile,ct_scan_90th_percentile,hip_fracture_repair_90th_percentile,hip_fracture_repair_emergency_inpatient_90th_percentile,hip_replacement_90th_percentile,knee_replacement_90th_percentile,lung_cancer_surgery_90th_percentile,mri_scan_90th_percentile,prostate_cancer_surgery_90th_percentile,radiation_therapy_90th_percentile
333,NS,Nova Scotia,2025,30.0,23.0,8.0,63.0,28.0,45.0,22.716667,...,62.0,248.0,51.681667,NaN,343.0,445.0,57.0,294.0,170.0,31.0
334,NB,New Brunswick,2025,29.0,24.0,11.0,103.0,22.0,NaN,19.683333,...,48.0,NaN,61.373333,NaN,397.0,407.0,45.0,NaN,139.0,27.0
344,AB,Alberta,2025,30.0,27.0,7.6,72.0,26.0,32.0,21.508333,...,60.0,143.0,54.356667,65.001667,610.0,605.0,61.0,220.0,174.0,23.0
345,SK,Saskatchewan,2025,22.0,19.0,6.0,63.0,21.0,21.0,33.725000,...,43.0,82.0,65.233333,NaN,477.0,494.0,35.0,178.0,85.0,22.0
347,NL,Newfoundland and Labrador,2025,NaN,NaN,9.0,54.0,NaN,NaN,23.400000,...,NaN,NaN,49.680000,NaN,665.0,842.0,NaN,NaN,NaN,37.0
348,PE,Prince Edward Island,2025,27.0,28.5,NaN,94.0,23.0,12.1,20.266667,...,37.0,88.4,52.966667,44.800000,515.0,805.5,NaN,527.5,NaN,23.0
349,ON,Ontario,2025,26.0,20.0,8.0,68.0,20.0,6.0,23.500000,...,38.0,116.0,54.201667,62.040000,300.0,337.0,46.0,187.0,95.0,18.2
405,QC,Quebec,2025,29.0,24.0,NaN,81.0,25.0,NaN,NaN,...,47.0,NaN,NaN,NaN,560.1,633.2,55.0,NaN,96.0,NaN
407,MB,Manitoba,2025,50.0,27.0,11.0,99.0,34.0,60.0,19.916667,...,72.0,236.0,55.826667,NaN,513.0,667.0,67.0,360.0,162.0,28.0
560,BC,British Columbia,2025,24.0,20.0,12.0,35.0,20.0,28.0,33.750000,...,47.0,243.0,69.535000,NaN,461.0,547.0,71.0,240.0,119.0,28.0


In [ ]:
df_access_2025["wait_time_indicators_available"] = (
    df_access_2025[wait_time_columns]
    .notna()
    .sum(axis=1)
)

df_access_2025[
    ["province_code", "province_name", "wait_time_indicators_available"]
].sort_values("wait_time_indicators_available", ascending=False)

,province_code,province_name,wait_time_indicators_available
349,ON,Ontario,28
344,AB,Alberta,28
560,BC,British Columbia,26
333,NS,Nova Scotia,26
407,MB,Manitoba,26
345,SK,Saskatchewan,26
334,NB,New Brunswick,22
348,PE,Prince Edward Island,22
405,QC,Quebec,16
347,NL,Newfoundland and Labrador,12


In [ ]:
wait_median_columns = [
    col for col in wait_time_columns
    if col.endswith("_median")
]

wait_90_columns = [
    col for col in wait_time_columns
    if col.endswith("_90th_percentile")
]

df_access_2025_long = df_access_2025.melt(
    id_vars=["province_code", "province_name", "data_year"],
    value_vars=wait_median_columns + wait_90_columns,
    var_name="wait_time_indicator",
    value_name="wait_days"
)

In [ ]:
df_access_median_2025 = df_access_2025[
    [
        "province_code",
        "province_name",
        "data_year"
    ] + wait_median_columns
].copy()

In [ ]:
df_access_90_2025 = df_access_2025[
    [
        "province_code",
        "province_name",
        "data_year"
    ] + wait_90_columns
].copy()

In [ ]:
df_access_median_2025.describe()

,data_year,bladder_cancer_surgery_median,breast_cancer_surgery_median,cabg_median,cataract_surgery_median,colorectal_cancer_surgery_median,ct_scan_median,hip_fracture_repair_median,hip_fracture_repair_emergency_inpatient_median,hip_replacement_median,knee_replacement_median,lung_cancer_surgery_median,mri_scan_median,prostate_cancer_surgery_median,radiation_therapy_median
count,10.0,9.000000,9.000000,8.000000,10.00000,9.000000,7.000000,9.000000,3.000000,10.000000,10.000000,8.000000,7.000000,8.000000,9.000000
mean,2025.0,29.666667,23.611111,9.075000,73.20000,24.333333,29.157143,24.274074,25.822222,227.750000,283.600000,22.375000,74.271429,55.750000,14.911111
std,0.0,8.108637,3.443996,2.067262,21.37392,4.555217,18.732490,5.552842,7.131879,69.452322,96.674942,3.662064,37.598525,17.417151,4.378483
min,2025.0,22.000000,19.000000,6.000000,35.00000,20.000000,6.000000,19.683333,17.600000,123.000000,133.000000,16.000000,41.900000,40.000000,11.200000
25%,2025.0,26.000000,20.000000,7.900000,63.00000,21.000000,16.550000,20.266667,23.566667,176.750000,203.750000,20.000000,49.000000,42.250000,12.000000
50%,2025.0,29.000000,24.000000,8.500000,70.00000,23.000000,28.000000,22.716667,29.533333,249.750000,293.000000,23.000000,68.000000,47.000000,13.000000
75%,2025.0,30.000000,27.000000,11.000000,90.75000,26.000000,38.500000,23.500000,29.933333,270.500000,360.250000,24.250000,80.500000,74.750000,17.000000
max,2025.0,50.000000,28.500000,12.000000,103.00000,34.000000,60.000000,33.750000,30.333333,337.000000,424.000000,28.000000,151.000000,78.000000,23.000000


#### **4.1.1 Province with the Highest and Lowest Median Wait times for Hip Replacement**

In [ ]:
df_access_median_2025[
    [
        "province_code",
        "province_name",
        "hip_replacement_median"
    ]
].sort_values("hip_replacement_median", ascending=False)

,province_code,province_name,hip_replacement_median
407,MB,Manitoba,337.0
344,AB,Alberta,281.0
348,PE,Prince Edward Island,273.0
405,QC,Quebec,263.0
347,NL,Newfoundland and Labrador,253.0
345,SK,Saskatchewan,246.5
334,NB,New Brunswick,200.0
560,BC,British Columbia,169.0
333,NS,Nova Scotia,132.0
349,ON,Ontario,123.0


In [ ]:
results = []

for column in wait_median_columns:

    values = df_access_median_2025[
        ["province_code", "province_name", column]
    ].dropna()

    highest = values.loc[values[column].idxmax()]
    lowest = values.loc[values[column].idxmin()]

    results.append({
        "service": column.replace("_median", ""),
        "highest_wait_province": highest["province_name"],
        "highest_wait_days": highest[column],
        "lowest_wait_province": lowest["province_name"],
        "lowest_wait_days": lowest[column]
    })

df_access_median_summary = pd.DataFrame(results)

df_access_median_summary

,service,highest_wait_province,highest_wait_days,lowest_wait_province,lowest_wait_days
0,bladder_cancer_surgery,Manitoba,50.000000,Saskatchewan,22.000000
1,breast_cancer_surgery,Prince Edward Island,28.500000,Saskatchewan,19.000000
2,cabg,British Columbia,12.000000,Saskatchewan,6.000000
3,cataract_surgery,New Brunswick,103.000000,British Columbia,35.000000
4,colorectal_cancer_surgery,Manitoba,34.000000,Ontario,20.000000
5,ct_scan,Manitoba,60.000000,Ontario,6.000000
6,hip_fracture_repair,British Columbia,33.750000,New Brunswick,19.683333
7,hip_fracture_repair_emergency_inpatient,Alberta,30.333333,Prince Edward Island,17.600000
8,hip_replacement,Manitoba,337.000000,Ontario,123.000000
9,knee_replacement,Prince Edward Island,424.000000,Ontario,133.000000


> ##### **Insights:** 
> - ##### The results above shows that in 2025, healthcare wait times varied substantially across Canadian provinces and across service categories. Median waits were highest in Manitoba for several services, including hip replacement (337 days), MRI (151 days), and CT scans (60 days), while Prince Edward Island recorded the highest median waits for knee replacement (424 days) and breast cancer surgery (28.5 days). Ontario and Saskatchewan frequently recorded the lowest median waits, although no province consistently had the shortest wait across all services. These differences indicate considerable variation in access pressure across healthcare services and jurisdictions.

In [ ]:
results = []

for column in wait_90_columns:

    values = df_access_90_2025[
        ["province_code", "province_name", column]
    ].dropna()

    highest = values.loc[values[column].idxmax()]
    lowest = values.loc[values[column].idxmin()]

    results.append({
        "service": column.replace("_90th_percentile", ""),
        "highest_wait_province": highest["province_name"],
        "highest_wait_days": highest[column],
        "lowest_wait_province": lowest["province_name"],
        "lowest_wait_days": lowest[column]
    })

df_access_90_summary = pd.DataFrame(results)

df_access_90_summary

,service,highest_wait_province,highest_wait_days,lowest_wait_province,lowest_wait_days
0,bladder_cancer_surgery,Nova Scotia,98.000000,Saskatchewan,41.00
1,breast_cancer_surgery,Alberta,57.000000,Ontario,36.00
2,cabg,Newfoundland and Labrador,196.600000,Saskatchewan,15.00
3,cataract_surgery,New Brunswick,472.000000,Nova Scotia,147.00
4,colorectal_cancer_surgery,Manitoba,72.000000,Prince Edward Island,37.00
5,ct_scan,Nova Scotia,248.000000,Saskatchewan,82.00
6,hip_fracture_repair,British Columbia,69.535000,Newfoundland and Labrador,49.68
7,hip_fracture_repair_emergency_inpatient,Alberta,65.001667,Prince Edward Island,44.80
8,hip_replacement,Newfoundland and Labrador,665.000000,Ontario,300.00
9,knee_replacement,Newfoundland and Labrador,842.000000,Ontario,337.00


> ##### **Insights:** 
> - ##### The 90th-percentile results reveal additional access pressure at the longer end of the waiting-time distribution, with Newfoundland and Labrador recording particularly high waits for knee replacement (842 days), hip replacement (665 days), and CABG (196.6 days). Overall, the results indicate substantial variation in access pressure across provinces and healthcare services, with no single province consistently recording the highest or lowest waits across all indicators.

### **4.2 Changes in Wait-time Overtime**

In [ ]:
df_access[wait_time_columns].notna().sum().sort_values(ascending=False)

cataract_surgery_median                                    194
knee_replacement_median                                    194
hip_replacement_median                                     194
cataract_surgery_90th_percentile                           194
knee_replacement_90th_percentile                           194
hip_replacement_90th_percentile                            194
radiation_therapy_median                                   170
radiation_therapy_90th_percentile                          168
hip_fracture_repair_90th_percentile                        167
hip_fracture_repair_median                                 167
cabg_median                                                157
cabg_90th_percentile                                       157
breast_cancer_surgery_median                               142
breast_cancer_surgery_90th_percentile                      142
bladder_cancer_surgery_90th_percentile                     131
colorectal_cancer_surgery_90th_percentile              

In [ ]:
df_access[
    [
        "data_year",
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
].dropna(
    subset=[
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
).sort_values("data_year")

,data_year,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
625,2008,103.000000,70.000000,41.000000,10.000000
715,2008,110.000000,78.000000,79.000000,22.000000
624,2008,63.000000,44.000000,30.000000,9.000000
671,2009,93.000000,88.000000,57.000000,11.000000
713,2009,178.000000,67.000000,48.000000,18.000000
...,...,...,...,...,...
348,2025,273.000000,94.000000,41.900000,12.100000
349,2025,123.000000,68.000000,47.000000,6.000000
407,2025,337.000000,99.000000,151.000000,60.000000
354,2025,120.116889,66.408943,59.390738,14.739853


In [ ]:
df_access_trends = df_access[
    (df_access["data_year"].between(2008, 2025))
][
    [
        "province_code",
        "province_name",
        "data_year",
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
].copy()

In [ ]:
df_access_trends

,province_code,province_name,data_year,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
0,PE,Prince Edward Island,2024,291.0,352.6,78.0,18.1
3,BC,British Columbia,2022,247.0,44.0,58.0,24.0
4,NB,New Brunswick,2021,197.0,71.0,NaN,NaN
8,QC,Quebec,2020,347.0,99.0,NaN,NaN
9,ON,Ontario,2016,80.0,65.0,33.0,6.0
...,...,...,...,...,...,...,...
722,YT,Yukon,2011,NaN,NaN,NaN,NaN
723,MB,Manitoba,2018,367.0,167.0,56.0,20.0
724,AB,Alberta,2016,138.0,92.0,85.0,27.0
725,SK,Saskatchewan,2015,63.0,35.0,32.0,21.0


In [ ]:
df_access_trend_change = (
    df_access_trends[
        df_access_trends["data_year"].isin([2008, 2025])
    ]
    .sort_values(["province_code", "data_year"])
    .groupby(["province_code", "province_name"])
    .agg(
        hip_replacement_start=("hip_replacement_median", "first"),
        hip_replacement_end=("hip_replacement_median", "last"),
        cataract_start=("cataract_surgery_median", "first"),
        cataract_end=("cataract_surgery_median", "last"),
        mri_start=("mri_scan_median", "first"),
        mri_end=("mri_scan_median", "last"),
        ct_start=("ct_scan_median", "first"),
        ct_end=("ct_scan_median", "last")
    )
    .reset_index()
)
df_access_trend_change

,province_code,province_name,hip_replacement_start,hip_replacement_end,cataract_start,cataract_end,mri_start,mri_end,ct_start,ct_end
0,AB,Alberta,103.000000,281.000000,70.000000,72.000000,41.000000,72.000000,10.000000,32.000000
1,BC,British Columbia,71.000000,169.000000,55.000000,35.000000,89.000000,89.000000,28.000000,28.000000
2,CA,Canada,120.116889,120.116889,66.408943,66.408943,59.390738,59.390738,14.739853,14.739853
3,MB,Manitoba,139.000000,337.000000,75.000000,99.000000,151.000000,151.000000,60.000000,60.000000
4,NB,New Brunswick,140.000000,200.000000,57.000000,103.000000,NaN,NaN,NaN,NaN
5,NL,Newfoundland and Labrador,253.000000,253.000000,54.000000,54.000000,NaN,NaN,NaN,NaN
6,NS,Nova Scotia,201.000000,132.000000,54.000000,63.000000,68.000000,68.000000,45.000000,45.000000
7,NT,Northwest Territories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NU,Nunavut,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ON,Ontario,63.000000,123.000000,44.000000,68.000000,30.000000,47.000000,9.000000,6.000000


In [ ]:
df_access_trend_change["hip_replacement_change_pct"] = (
    (df_access_trend_change["hip_replacement_end"]
    - df_access_trend_change["hip_replacement_start"])
    / df_access_trend_change["hip_replacement_start"]
) * 100

In [ ]:
df_access_trend_change["cataract_change_pct"] = (
    (df_access_trend_change["cataract_end"]
     - df_access_trend_change["cataract_start"])
     / df_access_trend_change["cataract_start"]
) * 100

In [ ]:
df_access_trend_change["mri_change_pct"] = (
    (df_access_trend_change["mri_end"]
     - df_access_trend_change["mri_start"])
     / df_access_trend_change["mri_start"]
) * 100

In [ ]:
df_access_trend_change["ct_change_pct"] = (
    (df_access_trend_change["ct_end"]
     - df_access_trend_change["ct_start"])
    / df_access_trend_change["ct_start"]
) * 100

In [ ]:
df_access_trend_change

,province_code,province_name,hip_replacement_start,hip_replacement_end,cataract_start,cataract_end,mri_start,mri_end,ct_start,ct_end,hip_replacement_change_pct,cataract_change_pct,mri_change_pct,ct_change_pct
0,AB,Alberta,103.000000,281.000000,70.000000,72.000000,41.000000,72.000000,10.000000,32.000000,172.815534,2.857143,75.609756,220.000000
1,BC,British Columbia,71.000000,169.000000,55.000000,35.000000,89.000000,89.000000,28.000000,28.000000,138.028169,-36.363636,0.000000,0.000000
2,CA,Canada,120.116889,120.116889,66.408943,66.408943,59.390738,59.390738,14.739853,14.739853,0.000000,0.000000,0.000000,0.000000
3,MB,Manitoba,139.000000,337.000000,75.000000,99.000000,151.000000,151.000000,60.000000,60.000000,142.446043,32.000000,0.000000,0.000000
4,NB,New Brunswick,140.000000,200.000000,57.000000,103.000000,NaN,NaN,NaN,NaN,42.857143,80.701754,NaN,NaN
5,NL,Newfoundland and Labrador,253.000000,253.000000,54.000000,54.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN
6,NS,Nova Scotia,201.000000,132.000000,54.000000,63.000000,68.000000,68.000000,45.000000,45.000000,-34.328358,16.666667,0.000000,0.000000
7,NT,Northwest Territories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NU,Nunavut,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ON,Ontario,63.000000,123.000000,44.000000,68.000000,30.000000,47.000000,9.000000,6.000000,95.238095,54.545455,56.666667,-33.333333


In [ ]:
df_access_trend_change[
    [
        "province_code",
        "province_name",
        "hip_replacement_change_pct",
        "cataract_change_pct",
        "mri_change_pct",
        "ct_change_pct"
    ]
].round(1)

,province_code,province_name,hip_replacement_change_pct,cataract_change_pct,mri_change_pct,ct_change_pct
0,AB,Alberta,172.8,2.9,75.6,220.0
1,BC,British Columbia,138.0,-36.4,0.0,0.0
2,CA,Canada,0.0,0.0,0.0,0.0
3,MB,Manitoba,142.4,32.0,0.0,0.0
4,NB,New Brunswick,42.9,80.7,NaN,NaN
5,NL,Newfoundland and Labrador,0.0,0.0,NaN,NaN
6,NS,Nova Scotia,-34.3,16.7,0.0,0.0
7,NT,Northwest Territories,NaN,NaN,NaN,NaN
8,NU,Nunavut,NaN,NaN,NaN,NaN
9,ON,Ontario,95.2,54.5,56.7,-33.3


> ##### **Insights:** 
> - ##### Hip replacement wait times increased in 8 of 10 provinces. The largest increases were in Quebec (+281.2%), Alberta (+172.8%), Manitoba (+142.4%), and PEI (+148.2%). Wait times decreased only in Nova Scotia (−34.3%) and were unchanged in Newfoundland and Labrador.
> - ##### Cataract surgery was more mixed. Wait times increased in Alberta, Manitoba, New Brunswick, Newfoundland and Labrador, Nova Scotia, Ontario and PEI, while decreasing in British Columbia (−36.4%) and Saskatchewan (−46.6%). Quebec had the largest increase at +92.9%.
> - ##### Diagnostic access was mixed where comparable data are available. MRI wait times increased in Alberta (+75.6%) and Ontario (+56.7%), while decreasing in PEI (−47.0%); several provinces showed no change in the reported endpoint values. CT wait times increased substantially in Alberta (+220.0%) but decreased in Ontario (−33.3%) and PEI (−45.0%).

### **4.3 Provinces With The Greatest Pressure**

In [ ]:
df_access_median_summary

,service,highest_wait_province,highest_wait_days,lowest_wait_province,lowest_wait_days
0,bladder_cancer_surgery,Manitoba,50.000000,Saskatchewan,22.000000
1,breast_cancer_surgery,Prince Edward Island,28.500000,Saskatchewan,19.000000
2,cabg,British Columbia,12.000000,Saskatchewan,6.000000
3,cataract_surgery,New Brunswick,103.000000,British Columbia,35.000000
4,colorectal_cancer_surgery,Manitoba,34.000000,Ontario,20.000000
5,ct_scan,Manitoba,60.000000,Ontario,6.000000
6,hip_fracture_repair,British Columbia,33.750000,New Brunswick,19.683333
7,hip_fracture_repair_emergency_inpatient,Alberta,30.333333,Prince Edward Island,17.600000
8,hip_replacement,Manitoba,337.000000,Ontario,123.000000
9,knee_replacement,Prince Edward Island,424.000000,Ontario,133.000000


In [ ]:
df_access_90_summary

,service,highest_wait_province,highest_wait_days,lowest_wait_province,lowest_wait_days
0,bladder_cancer_surgery,Nova Scotia,98.000000,Saskatchewan,41.00
1,breast_cancer_surgery,Alberta,57.000000,Ontario,36.00
2,cabg,Newfoundland and Labrador,196.600000,Saskatchewan,15.00
3,cataract_surgery,New Brunswick,472.000000,Nova Scotia,147.00
4,colorectal_cancer_surgery,Manitoba,72.000000,Prince Edward Island,37.00
5,ct_scan,Nova Scotia,248.000000,Saskatchewan,82.00
6,hip_fracture_repair,British Columbia,69.535000,Newfoundland and Labrador,49.68
7,hip_fracture_repair_emergency_inpatient,Alberta,65.001667,Prince Edward Island,44.80
8,hip_replacement,Newfoundland and Labrador,665.000000,Ontario,300.00
9,knee_replacement,Newfoundland and Labrador,842.000000,Ontario,337.00


> ##### **Insights:** 
> - ##### Access pressure varies substantially across Canadian provinces and healthcare services. Manitoba records the highest median wait for five of the 14 services examined, indicating relatively high typical waiting times across several service categories. At the longer end of the waiting distribution, Newfoundland and Labrador records the highest 90th-percentile wait for four services, including CABG, hip replacement, knee replacement, and radiation therapy. Alberta, British Columbia, New Brunswick, and Prince Edward Island also appear repeatedly among the provinces with the highest waits across multiple indicators. Overall, the evidence indicates that access pressure is widespread but differs by service and by position within the waiting-time distribution, with no single province consistently experiencing the highest waits across all measures.

### **4.4  Relationship Between Healthcare Capacity and Wait-times Recorded Across Provinces in 2022.**

> - ##### Period of analysis vary in this section because of data availability. 

In [ ]:
df_access[
    df_access["data_year"] == 2022
    ][
        [
        "province_code",
        "province_name",
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
]

,province_code,province_name,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
3,BC,British Columbia,247.0000,44.000000,58.00000,24.000000
33,PE,Prince Edward Island,201.6000,220.500000,41.20000,24.300000
44,NT,Northwest Territories,NaN,NaN,NaN,NaN
59,NB,New Brunswick,279.0000,79.000000,NaN,NaN
75,MB,Manitoba,300.0000,126.000000,134.00000,37.000000
130,NS,Nova Scotia,497.0000,98.000000,91.00000,47.000000
131,SK,Saskatchewan,367.0000,77.000000,49.00000,31.000000
314,AB,Alberta,267.0000,77.000000,53.00000,20.000000
490,QC,Quebec,280.0000,65.000000,NaN,NaN
516,NL,Newfoundland and Labrador,205.0000,154.000000,NaN,NaN


In [ ]:
#### Please refer to notebook

query = """
SELECT
    Province_id,
    province_code,
    province_name,
    data_year,
    population,
    population_65_plus,
    population_80_plus,
    population_85_plus,
    population_65_share_pct,
    population_80_share_pct,
    population_85_share_pct,
    hospital_beds_total,
    current_hospital_beds_total,
    physicians_total,
    registered_nurses,
    licensed_practical_nurses,
    nurse_practitioners,
    registered_psychiatric_nurses,
    icu_beds,
    long_term_care_beds,
    mental_health_addictions_beds,
    obstetrics_beds,
    other_acute_care_beds,
    pediatrics_beds,
    rated_capacity_beds,
    rehabilitation_beds
FROM master.analytics_province_year
WHERE province_code NOT IN ('CA', 'NT', 'YT', 'NU')
ORDER BY province_name, data_year;
"""

df_healthcare_capacity = pd.read_sql(query, engine)

#### Changes in Bed Capacity Across Provinces
df_bed_capacity_change = (df_healthcare_capacity[
  df_healthcare_capacity["hospital_beds_total"].notna()
].groupby(
    ["province_code", "province_name"]
).agg(
    start_year=("data_year", "min"),
    end_year=("data_year", "max"),
    start_beds=("hospital_beds_total", "first"),
    end_beds=("hospital_beds_total", "last")
).reset_index()
)

##### Absolute Change in Bed Capacity
df_bed_capacity_change["bed_change_absolute"] = (
    df_bed_capacity_change["end_beds"] - df_bed_capacity_change["start_beds"]
)


 #### Changes in Bed Capacity Across Provinces
df_bed_capacity_change["bed_change_pct"] = (
    df_bed_capacity_change["bed_change_absolute"]
    / df_bed_capacity_change["start_beds"]
) * 100

df_healthcare_capacity["beds_per_100k"] = (
    df_healthcare_capacity["hospital_beds_total"]
    / df_healthcare_capacity["population"]
) * 100000


df_healthcare_capacity[
    df_healthcare_capacity["beds_per_100k"].notna()
].sort_values(
    ["province_code", "data_year"]
).head()

df_healthcare_capacity["physicians_per_100k"] = (
    df_healthcare_capacity["physicians_total"]
    /df_healthcare_capacity["population"]
) * 100000

In [ ]:
df_healthcare_capacity.columns.tolist()

['province_id',
 'province_code',
 'province_name',
 'data_year',
 'population',
 'population_65_plus',
 'population_80_plus',
 'population_85_plus',
 'population_65_share_pct',
 'population_80_share_pct',
 'population_85_share_pct',
 'hospital_beds_total',
 'current_hospital_beds_total',
 'physicians_total',
 'registered_nurses',
 'licensed_practical_nurses',
 'nurse_practitioners',
 'registered_psychiatric_nurses',
 'icu_beds',
 'long_term_care_beds',
 'mental_health_addictions_beds',
 'obstetrics_beds',
 'other_acute_care_beds',
 'pediatrics_beds',
 'rated_capacity_beds',
 'rehabilitation_beds',
 'beds_per_100k',
 'physicians_per_100k']

In [ ]:
df_capacity_2022 = (
    df_healthcare_capacity[
        (df_healthcare_capacity["data_year"] == 2022)
    ][
        [
            "province_code",
            "province_name",
            "beds_per_100k"
        ]
    ]
)

In [ ]:
df_access_2022 = (
    df_access[
        (df_access["data_year"] == 2022)       
    ][
        [
            "province_code",
            "province_name",
            "hip_replacement_median",
            "cataract_surgery_median",
            "mri_scan_median",
            "ct_scan_median"
        ]
    ]
)

In [ ]:
df_access_2022

,province_code,province_name,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
3,BC,British Columbia,247.0000,44.000000,58.00000,24.000000
33,PE,Prince Edward Island,201.6000,220.500000,41.20000,24.300000
44,NT,Northwest Territories,NaN,NaN,NaN,NaN
59,NB,New Brunswick,279.0000,79.000000,NaN,NaN
75,MB,Manitoba,300.0000,126.000000,134.00000,37.000000
130,NS,Nova Scotia,497.0000,98.000000,91.00000,47.000000
131,SK,Saskatchewan,367.0000,77.000000,49.00000,31.000000
314,AB,Alberta,267.0000,77.000000,53.00000,20.000000
490,QC,Quebec,280.0000,65.000000,NaN,NaN
516,NL,Newfoundland and Labrador,205.0000,154.000000,NaN,NaN


In [ ]:
df_capacity_access = df_capacity_2022.merge(
    df_access_2022,
    on=["province_code", "province_name"],
    how="inner"
    
)

In [ ]:
df_capacity_access

,province_code,province_name,beds_per_100k,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
0,AB,Alberta,240.630341,267.0,77.0,53.0,20.0
1,BC,British Columbia,232.494129,247.0,44.0,58.0,24.0
2,MB,Manitoba,295.578770,300.0,126.0,134.0,37.0
3,NB,New Brunswick,315.502263,279.0,79.0,NaN,NaN
4,NL,Newfoundland and Labrador,411.100465,205.0,154.0,NaN,NaN
5,NS,Nova Scotia,316.493398,497.0,98.0,91.0,47.0
6,ON,Ontario,229.489155,108.0,87.0,35.0,9.0
7,PE,Prince Edward Island,287.679426,201.6,220.5,41.2,24.3
8,QC,Quebec,228.766836,280.0,65.0,NaN,NaN
9,SK,Saskatchewan,293.774326,367.0,77.0,49.0,31.0


In [ ]:
df_capacity_access[
    [
        "beds_per_100k",
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
].corr()

,beds_per_100k,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
beds_per_100k,1.000000,0.187698,0.497769,0.535010,0.869068
hip_replacement_median,0.187698,1.000000,-0.224561,0.500910,0.918537
cataract_surgery_median,0.497769,-0.224561,1.000000,0.022925,0.083921
mri_scan_median,0.535010,0.500910,0.022925,1.000000,0.718863
ct_scan_median,0.869068,0.918537,0.083921,0.718863,1.000000


> ##### **Insights:** 
> - ##### The correlation results above show that in year 2022, higher hospital-bed capacity per 100,000 population was not associated with shorter wait times cross-provincial comparison. The relationship was weakly positive for hip replacement and moderately to strongly positive for cataract surgery, MRI, and CT, with the strongest association observed for CT scans (r = 0.87). This indicates that hospital-bed capacity alone is not responsible for lower waiting times across the selected access indicators. The findings also suggest that access pressure reflects factors beyond aggregate hospital-bed capacity.

### **4.5 Relationship Between Expenditure and Wait-times recorded in 2022.**

In [ ]:
query = """
SELECT
    province_code,
    province_name,
    data_year,
    total_health_expenditure_current,
    total_health_expenditure_per_capita,
    total_health_expenditure_constant_2010,
    total_health_expenditure_constant_2010_per_capita,
    public_health_expenditure_current,
    public_health_expenditure_per_capita,
    public_health_expenditure_constant_2010,
    public_health_expenditure_constant_2010_per_capita,
    private_health_expenditure_current,
    private_health_expenditure_per_capita,
    private_health_expenditure_constant_2010,
    private_health_expenditure_constant_2010_per_capita,
    provincial_government_health_expenditure_current,
    provincial_government_health_expenditure_per_capita,
    provincial_government_health_expenditure_constant_2010,
    territorial_government_health_expenditure_current,
    territorial_government_health_expenditure_per_capita,
    territorial_government_health_expenditure_constant_2010
    FROM master.analytics_province_year
WHERE province_code NOT IN ('CA', 'NT', 'YT', 'NU')
ORDER BY province_name, data_year;
"""

df_expenditure = pd.read_sql(query, engine)

df_expenditure

,province_code,province_name,data_year,total_health_expenditure_current,total_health_expenditure_per_capita,total_health_expenditure_constant_2010,total_health_expenditure_constant_2010_per_capita,public_health_expenditure_current,public_health_expenditure_per_capita,public_health_expenditure_constant_2010,...,private_health_expenditure_current,private_health_expenditure_per_capita,private_health_expenditure_constant_2010,private_health_expenditure_constant_2010_per_capita,provincial_government_health_expenditure_current,provincial_government_health_expenditure_per_capita,provincial_government_health_expenditure_constant_2010,territorial_government_health_expenditure_current,territorial_government_health_expenditure_per_capita,territorial_government_health_expenditure_constant_2010
0,AB,Alberta,1971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None
1,AB,Alberta,1972,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None
2,AB,Alberta,1973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None
3,AB,Alberta,1974,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None
4,AB,Alberta,1975,9.923433e+08,548.653375,4.939117e+09,2730.771733,7.577722e+08,418.962157,3.738962e+09,...,2.345711e+08,129.691218,1.200154e+09,663.549287,6.949125e+08,384.207843,3.428803e+09,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
545,SK,Saskatchewan,2021,1.070319e+10,9165.958661,8.509617e+09,7287.434135,8.030554e+09,6877.175979,6.136982e+09,...,2.672637e+09,2288.782682,2.372635e+09,2031.867969,6.551220e+09,5610.309440,5.006469e+09,None,None,None
546,SK,Saskatchewan,2022,1.094586e+10,9288.316913,8.438291e+09,7160.469199,8.121616e+09,6891.748606,5.997589e+09,...,2.824248e+09,2396.568307,2.440702e+09,2071.102846,6.737869e+09,5717.544214,4.975730e+09,None,None,None
547,SK,Saskatchewan,2023,1.178260e+10,9743.268518,8.788776e+09,7267.613804,8.697784e+09,7192.370909,6.235791e+09,...,3.084818e+09,2550.897610,2.552985e+09,2111.114097,7.294317e+09,6031.815421,5.229589e+09,None,None,None
548,SK,Saskatchewan,2024,1.220294e+10,9842.150434,8.780179e+09,7081.560195,8.980860e+09,7243.417229,6.185453e+09,...,3.222078e+09,2598.733204,2.594726e+09,2092.748852,7.560850e+09,6098.123632,5.207439e+09,None,None,None


In [ ]:
df_expenditure_2022 = df_expenditure[
    (df_expenditure["data_year"] == 2022)
 ][
    [
        "province_code",
        "province_name",
        "total_health_expenditure_constant_2010_per_capita"
    ]
].copy()

In [ ]:
df_expenditure_access = df_expenditure_2022.merge(
    df_access_2022,
    on=["province_code", "province_name"],
    how="inner"
)

In [ ]:
df_expenditure_access

,province_code,province_name,total_health_expenditure_constant_2010_per_capita,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
0,AB,Alberta,7329.923567,267.0,77.0,53.0,20.0
1,BC,British Columbia,6965.802673,247.0,44.0,58.0,24.0
2,MB,Manitoba,6835.770473,300.0,126.0,134.0,37.0
3,NB,New Brunswick,6781.423381,279.0,79.0,NaN,NaN
4,NL,Newfoundland and Labrador,8167.953550,205.0,154.0,NaN,NaN
5,NS,Nova Scotia,7611.734757,497.0,98.0,91.0,47.0
6,ON,Ontario,6556.048428,108.0,87.0,35.0,9.0
7,PE,Prince Edward Island,6524.847359,201.6,220.5,41.2,24.3
8,QC,Quebec,6513.088385,280.0,65.0,NaN,NaN
9,SK,Saskatchewan,7160.469199,367.0,77.0,49.0,31.0


In [ ]:
df_expenditure_access[
    [
        "total_health_expenditure_constant_2010_per_capita",
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
].corr()

,total_health_expenditure_constant_2010_per_capita,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
total_health_expenditure_constant_2010_per_capita,1.000000,0.335812,0.068624,0.273026,0.621655
hip_replacement_median,0.335812,1.000000,-0.224561,0.500910,0.918537
cataract_surgery_median,0.068624,-0.224561,1.000000,0.022925,0.083921
mri_scan_median,0.273026,0.500910,0.022925,1.000000,0.718863
ct_scan_median,0.621655,0.918537,0.083921,0.718863,1.000000


> ##### **Insights:** 
> - ##### The correlation coefficient shows that real healthcare expenditure per capita did not correspond consistently with shorter healthcare wait times in the 2022 cross-provincial comparison. The relationship was weakly positive for hip replacement (r = 0.336) and MRI (r = 0.273), very weak for cataract surgery (r = 0.069), and moderately positive for CT scans (r = 0.622). These results indicate that higher healthcare expenditure per capita was not associated with lower waiting times across the selected access indicators. The findings also suggest that financial resources alone do not fully explain differences in healthcare access.

### **4.6 Relationship Between Changes in Population Size and Changes in Wait-times**

In [ ]:
query = """
    SELECT
    province_id,
    province_code,
    province_name,
    data_year,
    population,
    population_65_plus,
    population_80_plus,
    population_85_plus,
    population_65_share_pct,
    population_80_share_pct,
    population_85_share_pct
FROM master.analytics_province_year
WHERE province_code NOT IN ('CA', 'NT', 'YT', 'NU')
ORDER BY province_name, data_year;
"""

df_population_demand = pd.read_sql(query, engine)

df_population_demand

,province_id,province_code,province_name,data_year,population,population_65_plus,population_80_plus,population_85_plus,population_65_share_pct,population_80_share_pct,population_85_share_pct
0,6,AB,Alberta,1971,1665717.0,120453.0,25438.0,10431.0,7.231300,1.527150,0.626217
1,6,AB,Alberta,1972,1694090.0,123588.0,26188.0,10894.0,7.295244,1.545845,0.643059
2,6,AB,Alberta,1973,1725327.0,126985.0,26669.0,11424.0,7.360054,1.545736,0.662135
3,6,AB,Alberta,1974,1754621.0,130045.0,27048.0,11973.0,7.411572,1.541529,0.682370
4,6,AB,Alberta,1975,1808689.0,134156.0,27442.0,12282.0,7.417306,1.517232,0.679055
...,...,...,...,...,...,...,...,...,...,...,...
545,17,SK,Saskatchewan,2021,1167711.0,198565.0,50771.0,27399.0,17.004636,4.347908,2.346385
546,17,SK,Saskatchewan,2022,1178796.0,204647.0,50862.0,26970.0,17.360680,4.314741,2.287928
547,17,SK,Saskatchewan,2023,1210257.0,211510.0,51537.0,26972.0,17.476453,4.258352,2.228618
548,17,SK,Saskatchewan,2024,1247868.0,218531.0,52274.0,27014.0,17.512349,4.189065,2.164812


In [ ]:
df_population_2008_2022 = df_population_demand[
    (df_population_demand["data_year"].isin([2008, 2022]))
][
    [
        "province_code",
        "province_name",
        "data_year",
        "population"
    ]
].copy()

In [ ]:
df_population_2008_2022 = (
    df_population_2008_2022
    .pivot(
        index=["province_code", "province_name"],
        columns="data_year",
        values="population"
    )
    .reset_index()
)
df_population_2008_2022

data_year,province_code,province_name,2008,2022
0,AB,Alberta,3595875.0,4512731.0
1,BC,British Columbia,4349338.0,5358845.0
2,MB,Manitoba,1197767.0,1413498.0
3,NB,New Brunswick,746875.0,808869.0
4,NL,Newfoundland and Labrador,511569.0,531257.0
5,NS,Nova Scotia,935965.0,1024034.0
6,ON,Ontario,12883824.0,15155836.0
7,PE,Prince Edward Island,138736.0,167200.0
8,QC,Quebec,7761614.0,8669963.0
9,SK,Saskatchewan,1017368.0,1178796.0


In [ ]:
df_population_2008_2022["population_growth_pct"] = (
    (df_population_2008_2022[2022] 
    - df_population_2008_2022[2008])
    / df_population_2008_2022[2008]
) * 100

In [ ]:
df_wait_growth = (
    df_access_trends[
        df_access_trends["data_year"].isin([2008, 2022])
    ]
    .sort_values(["province_code", "data_year"])
    [
        [
            "province_code",
            "province_name",
            "data_year",
            "hip_replacement_median",
            "cataract_surgery_median",
            "mri_scan_median",
            "ct_scan_median"
        ]
    ]
)

In [ ]:
df_wait_growth = (
    df_access_trends[
        df_access_trends["data_year"].isin([2008, 2022])
    ]
    .sort_values(["province_code", "data_year"])
    .groupby(["province_code", "province_name"])
    .agg(
        hip_replacement_2008=("hip_replacement_median", "first"),
        hip_replacement_2022=("hip_replacement_median", "last"),
        cataract_surgery_2008=("cataract_surgery_median", "first"),
        cataract_surgery_2022=("cataract_surgery_median", "last"),
        mri_scan_2008=("mri_scan_median", "first"),
        mri_scan_2022=("mri_scan_median", "last"),
        ct_scan_2008=("ct_scan_median", "first"),
        ct_scan_2022=("ct_scan_median", "last")
    )
    .reset_index()
)

df_wait_growth

,province_code,province_name,hip_replacement_2008,hip_replacement_2022,cataract_surgery_2008,cataract_surgery_2022,mri_scan_2008,mri_scan_2022,ct_scan_2008,ct_scan_2022
0,AB,Alberta,103.0000,267.0000,70.000000,77.000000,41.00000,53.00000,10.000000,20.000000
1,BC,British Columbia,71.0000,247.0000,55.000000,44.000000,58.00000,58.00000,24.000000,24.000000
2,CA,Canada,164.3255,164.3255,73.237508,73.237508,44.31798,44.31798,15.336759,15.336759
3,MB,Manitoba,139.0000,300.0000,75.000000,126.000000,134.00000,134.00000,37.000000,37.000000
4,NB,New Brunswick,140.0000,279.0000,57.000000,79.000000,NaN,NaN,NaN,NaN
5,NL,Newfoundland and Labrador,205.0000,205.0000,154.000000,154.000000,NaN,NaN,NaN,NaN
6,NS,Nova Scotia,201.0000,497.0000,54.000000,98.000000,91.00000,91.00000,47.000000,47.000000
7,NT,Northwest Territories,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NU,Nunavut,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ON,Ontario,63.0000,108.0000,44.000000,87.000000,30.00000,35.00000,9.000000,9.000000


In [ ]:
df_wait_growth["hip_replacement_growth_pct"] = (
    (df_wait_growth["hip_replacement_2022"] - df_wait_growth["hip_replacement_2008"])
    / df_wait_growth["hip_replacement_2008"]
) * 100

In [ ]:
df_wait_growth["cataract_surgery_growth_pct"] = (
    (df_wait_growth["cataract_surgery_2022"] - df_wait_growth["cataract_surgery_2008"])
    / df_wait_growth["cataract_surgery_2008"]
) * 100

In [ ]:
df_wait_growth["mri_scan_growth_pct"] = (
    (df_wait_growth["mri_scan_2022"] - df_wait_growth["mri_scan_2008"])
    / df_wait_growth["mri_scan_2008"]
) * 100

In [ ]:
df_wait_growth["ct_scan_growth_pct"] = (
    (df_wait_growth["ct_scan_2022"] - df_wait_growth["ct_scan_2008"])
    / df_wait_growth["ct_scan_2008"]
) * 100

In [ ]:
df_wait_growth[
    [
        "province_code",
        "province_name",
        "hip_replacement_growth_pct",
        "cataract_surgery_growth_pct",
        "mri_scan_growth_pct",
        "ct_scan_growth_pct"

    ]
]

,province_code,province_name,hip_replacement_growth_pct,cataract_surgery_growth_pct,mri_scan_growth_pct,ct_scan_growth_pct
0,AB,Alberta,159.223301,10.000000,29.268293,100.000000
1,BC,British Columbia,247.887324,-20.000000,0.000000,0.000000
2,CA,Canada,0.000000,0.000000,0.000000,0.000000
3,MB,Manitoba,115.827338,68.000000,0.000000,0.000000
4,NB,New Brunswick,99.285714,38.596491,NaN,NaN
5,NL,Newfoundland and Labrador,0.000000,0.000000,NaN,NaN
6,NS,Nova Scotia,147.263682,81.481481,0.000000,0.000000
7,NT,Northwest Territories,NaN,NaN,NaN,NaN
8,NU,Nunavut,NaN,NaN,NaN,NaN
9,ON,Ontario,71.428571,97.727273,16.666667,0.000000


In [ ]:
df_population_access_growth =df_population_2008_2022[
    [
        "province_code",
        "province_name",
        "population_growth_pct"
    ]
].merge(
    df_wait_growth[
        [
            "province_code",
            "province_name",
            "hip_replacement_growth_pct",
            "cataract_surgery_growth_pct",
            "mri_scan_growth_pct",
            "ct_scan_growth_pct"
        ]
    ],
    on=["province_code", "province_name"],
    how="inner"
)

In [ ]:
df_population_access_growth

,province_code,province_name,population_growth_pct,hip_replacement_growth_pct,cataract_surgery_growth_pct,mri_scan_growth_pct,ct_scan_growth_pct
0,AB,Alberta,25.497438,159.223301,10.000000,29.268293,100.000000
1,BC,British Columbia,23.210590,247.887324,-20.000000,0.000000,0.000000
2,MB,Manitoba,18.011099,115.827338,68.000000,0.000000,0.000000
3,NB,New Brunswick,8.300452,99.285714,38.596491,NaN,NaN
4,NL,Newfoundland and Labrador,3.848552,0.000000,0.000000,NaN,NaN
5,NS,Nova Scotia,9.409433,147.263682,81.481481,0.000000,0.000000
6,ON,Ontario,17.634609,71.428571,97.727273,16.666667,0.000000
7,PE,Prince Edward Island,20.516665,83.272727,182.692308,-47.848101,10.454545
8,QC,Quebec,11.703094,305.797101,54.761905,NaN,NaN
9,SK,Saskatchewan,15.867218,133.757962,-34.745763,0.000000,0.000000


In [ ]:
df_population_access_growth[
    [
        "population_growth_pct",
        "hip_replacement_growth_pct",
        "cataract_surgery_growth_pct",
        "mri_scan_growth_pct",
        "ct_scan_growth_pct"
    ]
    
].corr()

,population_growth_pct,hip_replacement_growth_pct,cataract_surgery_growth_pct,mri_scan_growth_pct,ct_scan_growth_pct
population_growth_pct,1.000000,0.324008,0.073012,0.124885,0.603504
hip_replacement_growth_pct,0.324008,1.000000,-0.208528,0.253801,0.127086
cataract_surgery_growth_pct,0.073012,-0.208528,1.000000,-0.616036,-0.185886
mri_scan_growth_pct,0.124885,0.253801,-0.616036,1.000000,0.459064
ct_scan_growth_pct,0.603504,0.127086,-0.185886,0.459064,1.000000


> ##### **Insights:** 
> - ##### The relationship between population growth and increasing healthcare access pressure varied considerably across services between 2008 and 2022. Population growth had a moderate positive correlation with growth in median CT wait times (r = 0.604) and a weaker positive correlation with hip-replacement wait-time growth (r = 0.324). The relationships were very weak for MRI (r = 0.125) and cataract surgery (r = 0.073). Overall, the results suggest that population growth coincided with greater increases in access pressure for some services, particularly CT scans, but the relationship was not consistent across healthcare services.

### **4.7 Provincial Resource Level and Wait time comparison**

In [ ]:
df_expenditure_access.sort_values("total_health_expenditure_constant_2010_per_capita")

,province_code,province_name,total_health_expenditure_constant_2010_per_capita,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
8,QC,Quebec,6513.088385,280.0,65.0,NaN,NaN
7,PE,Prince Edward Island,6524.847359,201.6,220.5,41.2,24.3
6,ON,Ontario,6556.048428,108.0,87.0,35.0,9.0
3,NB,New Brunswick,6781.423381,279.0,79.0,NaN,NaN
2,MB,Manitoba,6835.770473,300.0,126.0,134.0,37.0
1,BC,British Columbia,6965.802673,247.0,44.0,58.0,24.0
9,SK,Saskatchewan,7160.469199,367.0,77.0,49.0,31.0
0,AB,Alberta,7329.923567,267.0,77.0,53.0,20.0
5,NS,Nova Scotia,7611.734757,497.0,98.0,91.0,47.0
4,NL,Newfoundland and Labrador,8167.953550,205.0,154.0,NaN,NaN


In [ ]:
median_expenditure = (
    df_expenditure_access["total_health_expenditure_constant_2010_per_capita"].median()
)

median_expenditure

np.float64(6900.786573360216)

In [ ]:
df_expenditure_access["resource_difference_pct"] = (
    (df_expenditure_access["total_health_expenditure_constant_2010_per_capita"] - median_expenditure)
    / median_expenditure
 ) * 100

In [ ]:
df_expenditure_access[
    [
        "province_code",
        "province_name",
        "total_health_expenditure_constant_2010_per_capita",
        "resource_difference_pct"
    ]
].sort_values("resource_difference_pct").round(2)

,province_code,province_name,total_health_expenditure_constant_2010_per_capita,resource_difference_pct
8,QC,Quebec,6513.09,-5.62
7,PE,Prince Edward Island,6524.85,-5.45
6,ON,Ontario,6556.05,-5.00
3,NB,New Brunswick,6781.42,-1.73
2,MB,Manitoba,6835.77,-0.94
1,BC,British Columbia,6965.80,0.94
9,SK,Saskatchewan,7160.47,3.76
0,AB,Alberta,7329.92,6.22
5,NS,Nova Scotia,7611.73,10.30
4,NL,Newfoundland and Labrador,8167.95,18.36


In [ ]:
df_expenditure_access[
    df_expenditure_access["resource_difference_pct"].abs() <= 5
][
    [
        "province_code",
        "province_name",
        "total_health_expenditure_constant_2010_per_capita",
        "hip_replacement_median",
        "cataract_surgery_median",
        "mri_scan_median",
        "ct_scan_median"
    ]
].sort_values(
    "total_health_expenditure_constant_2010_per_capita"
).dropna()

,province_code,province_name,total_health_expenditure_constant_2010_per_capita,hip_replacement_median,cataract_surgery_median,mri_scan_median,ct_scan_median
6,ON,Ontario,6556.048428,108.0,87.0,35.0,9.0
2,MB,Manitoba,6835.770473,300.0,126.0,134.0,37.0
1,BC,British Columbia,6965.802673,247.0,44.0,58.0,24.0
9,SK,Saskatchewan,7160.469199,367.0,77.0,49.0,31.0


> ##### **Insights:** 
> - ##### Among provinces with relatively similar real healthcare expenditure per capita, substantial differences in healthcare wait times were still observed in 2022. The largest variation occurred in hip replacement, ranging from 108 days in Ontario to 367 days in Saskatchewan, while substantial differences were also observed for cataract surgery, MRI, and CT scans. Overall, the evidence indicates that healthcare expenditure per capita alone does not correspond closely with access outcomes, highlighting the importance of considering workforce, physical capacity, demographic demand, and other system characteristics when assessing healthcare access.


##### **Section Summary:**
> - ##### Healthcare access varies substantially across Canadian provinces and healthcare services, and differences in access do not correspond consistently with aggregate expenditure, hospital-bed capacity, or population growth. The evidence also shows that jurisdictions with relatively similar financial resource levels can experience substantially different waiting times, suggesting that access outcomes reflect a combination of resource availability, capacity, demographic demand, workforce, service configuration, and other system characteristics.

<table width="100%">
<tr>
<td width="50%" align="left"><a href="./03_RQ3_expenditure_resources.ipynb">← Previous: RQ3 — Expenditure Resources</a></td>
<td width="50%" align="right"><a href="./05_RQ5_healthcare_outcomes.ipynb">Next: RQ5 — Healthcare Outcomes →</a></td>
</tr>
</table>